In [3]:
import pandas as pd
import numpy as np

# read Parcels Tree Canopy Metric Dataset layer from Analyze Boston 
# Data reflects 2019 tree canopy area aggregated at the parcel level
parcels_tree_canopy_metrics = pd.read_csv('datasets/PARCELS_Tree_Canopy_Metrics.csv')

/var/folders/3k/5zd1hcfd4jv0bjvll1h2r7kr0000gn/T/ipykernel_52958/1077290031.py:6: DtypeWarning: Columns (4,6,12,14,18,27) have mixed types. Specify dtype option on import or set low_memory=False.
  parcels_tree_canopy_metrics = pd.read_csv('datasets/PARCELS_Tree_Canopy_Metrics.csv')


# 1. What percentage of Boston's tree canopy is located on public land versus residential parcels?

[Tree Canopy Metrics Dataset](https://bostonopendata-boston.opendata.arcgis.com/datasets/boston::canopy-change-assessment-tree-canopy-metrics/about?layer=4)

In [4]:
parcels_tree_canopy_metrics.head()

,FID,FID_Boston,BOSTON_LAN,FID_COB_FY,PID,CM_ID,GIS_ID,OWNER,ST_NUM,ST_NAME,...,TC_Pv_A,TC_Land_A,TC_Pi_A,TC_P_A,TC_E_P,TC_Pv_P,TC_P_P,TC_Pi_P,Shape__Area,Shape__Length
0,1,1,1,-1,,,,,,,...,5717593,218739022,38429015,44146608,18.670309,2.613888,20.182319,17.568431,2.244137e+08,8.914611e+06
1,2,1,1,1,0100001000,,0100001000,PASCUCCI CARLO,104 A 104,PUTNAM,...,0,1237,13,13,0.404204,0.000000,1.050930,1.050930,1.233845e+03,1.594376e+02
2,3,1,1,2,0100002000,,0100002000,ATANASOV DANIEL,197,LEXINGTON,...,0,1162,242,242,0.000000,0.000000,20.826162,20.826162,1.161436e+03,1.566642e+02
3,4,1,1,3,0100003000,,0100003000,CHEVARRIA ANA S,199,LEXINGTON,...,1,1144,211,212,0.699301,0.087413,18.531469,18.444056,1.145411e+03,1.558082e+02
4,5,1,1,4,0100004000,,0100004000,"MADDALENI JAMES E, TS",201,LEXINGTON,...,106,1191,102,208,0.587741,8.900084,17.464316,8.564232,1.192610e+03,1.571514e+02


In [5]:
parcels_tree_canopy_metrics.columns

Index(['FID', 'FID_Boston', 'BOSTON_LAN', 'FID_COB_FY', 'PID', 'CM_ID',
       'GIS_ID', 'OWNER', 'ST_NUM', 'ST_NAME', 'ST_NAME_SU', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'PTYPE', 'LOTSIZE', 'MAIL_ADDRE', 'MAIL_CS',
       'MAIL_ZIP', 'OWN_OCC', 'GROSS_AREA', 'BLDG_AREA', 'AV_TOTAL', 'AV_LAND',
       'AV_BLDG', 'GROSS_TAX', 'NUM_FLOORS', 'PID_LONG', 'FULL_ADDRE', 'CITY',
       'STArea__', 'STLength__', 'Shape_STAr', 'Shape_STLe', 'LU_GENERAL',
       'TC_ID', 'Shape_Leng', 'OBJECTID', 'TC_ID_1', 'Total_A', 'Can_A',
       'Grass_A', 'Soil_A', 'Water_A', 'Build_A', 'Road_A', 'Paved_A',
       'Perv_A', 'Imperv_A', 'Can_P', 'Grass_P', 'Soil_P', 'Water_P',
       'Build_P', 'Road_P', 'Paved_P', 'Perv_P', 'Imperv_P', 'OBJECTID_1',
       'TC_ID_12', 'VALUE_0', 'TC_E_A', 'TC_Pv_A', 'TC_Land_A', 'TC_Pi_A',
       'TC_P_A', 'TC_E_P', 'TC_Pv_P', 'TC_P_P', 'TC_Pi_P', 'Shape__Area',
       'Shape__Length'],
      dtype='object')

In [ ]:
parcels_tree_canopy_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173875 entries, 0 to 173874
Data columns (total 72 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   FID            173875 non-null  int64  
 1   FID_Boston     173875 non-null  int64  
 2   BOSTON_LAN     173875 non-null  int64  
 3   FID_COB_FY     173875 non-null  int64  
 4   PID            173875 non-null  object 
 5   CM_ID          173875 non-null  object 
 6   GIS_ID         173875 non-null  object 
 7   OWNER          173875 non-null  object 
 8   ST_NUM         173875 non-null  object 
 9   ST_NAME        173875 non-null  object 
 10  ST_NAME_SU     173875 non-null  object 
 11  UNIT_NUM       173875 non-null  object 
 12  ZIPCODE        173875 non-null  object 
 13  LU             173875 non-null  object 
 14  PTYPE          173875 non-null  object 
 15  LOTSIZE        173875 non-null  int64  
 16  MAIL_ADDRE     173875 non-null  object 
 17  MAIL_CS        173875 non-nul

## Formulas and Definitions for Calculations

[Tree Ordinance 2023](https://www.boston.gov/sites/default/files/file/2024/01/City%20of%20Boston%20Chapter%2014%20of%20the%20Ordinances%20of%202023%20-%20Ordinance%20Establishing%20Protections%20for%20the%20City%20of%20Boston%20Tree%20Canopy%20-%20City%20Clerk%20signature.pdf)

**Public Parcels:** is any parcel of land that is owned by the City of Boston. Includes,
* City of Boston
* Boston Water and Sewer Commission
* Boston Housing Authority
* Boston Public Schools
* Boston Redevelopment Authority
* Boston Planning and Development Agency

We included any OWNER that had "City of Boston" + "_", example "Fire Department"

**Private Parcels:** is any parcel of land that is not owned by the City of Boston

**Percentage of Tree Canopy of Public Parcels**  $$\frac{\text{Total Tree canopy existing area classifed as Public}}{\text{Total Tree canopy existing area}}$$


**Percentage of Tree Canopy of Private Parcels** $$\frac{\text{Total Tree canopy existing area classifed as Private}}{\text{Total Tree canopy existing area}}$$

**TC_E_A:** Tree canopy existing area. The area of tree canopy present when viewed from above using aerial or satellite imagery, excluding water.

## Exploring the Data

In [28]:
# count total number of unique owners 
len(parcels_tree_canopy_metrics["OWNER"].unique())

134787

In [27]:
# Count most common unique owners in the dataset
owner_count = parcels_tree_canopy_metrics['OWNER'].value_counts().to_frame(name='Count').reset_index()

# Standardize the names of the owners to be all uppercase
owner_count['OWNER'] = owner_count['OWNER'].str.upper()

# Show top 20 most common owners in Boston
owner_count.head(20)

,OWNER,Count
0,CITY OF BOSTON,2007
1,CITY OF BOSTON BY FCL,415
2,COMMONWEALTH OF MASS,226
3,TRUSTEES OF BOSTON COLLEGE,203
4,BOSTON HOUSING AUTHORITY,201
5,MARKETPLACE LOFTS LIMITED,162
6,COMMWLTH OF MASS,144
7,RELATED LOVEJOY RESIDENTIAL,133
8,BUCKMINSTER HOTEL CORP,132
9,BOSTON REDEVELOPMENT AUTH,132


In [7]:
# find all unique owners with the word 'Boston' in it
parcels_tree_canopy_metrics[parcels_tree_canopy_metrics['OWNER'].str.contains("BOSTON", case=False, na=False)]["OWNER"].unique()

array(['CITY OF BOSTON BY FCL', 'CITY OF BOSTON',
       'EAST BOSTON NEIGHBORHOOD', 'BOSTONIAN CONSTRUCTION',
       'EAST BOSTON AOP LLC', 'EXCEL ACADEMY E BOSTON RLTY',
       'BOSTON REDEVELOPMENT', 'EAST BOSTON SAVING BANK',
       'HVV EAST BOSTON LLC', 'ROMAN CATH ARCH BOSTON',
       '17 WHITE STREET BOSTON LLC', 'EAST BOSTON AREA PLANNING',
       'EAST BOSTON MANAGEMENT &', 'RRP EAST BOSTON ONE LLC',
       'BOSTON REDEVELOPMENT AUTHOR', 'BOSTON NATURAL AREAS FUND',
       'BOSTON COMMUNITY PROPERTIES', 'LINEAR RETAIL BOSTON #23',
       'LINEAR RETAIL BOSTON #21 LLC', 'LINEAR RETAIL BOSTON #19 LLC',
       'LINEAR RETAIL BOSTON #18 LLC', 'E BOSTON NEIGHBORHOOD HEALTH',
       'CITY OF BOSTON PARKS AND', 'EAST BOSTON MANAGEMENT',
       'BOSTON SAWYERS FAMILY TRUST', 'EAST BOSTON YACHT CLUB',
       'BOSTON MARINE WORKS', 'BOSTON MARINE WORKS INC',
       'EAST BOSTON COMMUNITY', 'EAST BOSTON AIRPORT LLC',
       'BOSTON MAJORDOMO LLC', 'CITY OF BOSTON PARKS COMM',
       'BO

## Data Cleaning

The dataset groups all ROW data together into one row, but left ownership as a blank space " " instead of null. Thus, we had to alter the dataset to encode this as ROW data because we do not know if it is entirely public or private. 

* Classified roads, sidewalks and streets as ROW data by setting the owner == "ROW"

In [ ]:
# Replace any rows with a ' '  owner with a null
parcels_tree_canopy_metrics["OWNER"] = parcels_tree_canopy_metrics["OWNER"].replace(" ", pd.NA)

# find all null rows
parcels_tree_canopy_metrics[parcels_tree_canopy_metrics["OWNER"].isna()]

# Classify the null row owner as 'ROW', since it represents all roads and sidewalks in Boston
# Streets can be public or private depending on ownership, but the Parcels_tree_canopy_metric data doesn't have unique records for each street
parcels_tree_canopy_metrics.loc[parcels_tree_canopy_metrics["OWNER"].isna(), "OWNER"] = "ROW"

## Data Analysis

We explored using three different methods of searching for all owners that fall into our definition of Public.
1. Baseline: Using String Matching Searches
2. Using RapidFuzz String Matching
3. Using Property Type Codes

## 1. Baseline: Using String Matching Searches

In [ ]:
#list of owners that are classified as publicly owned
#obtained by using str.contains outputs
public_owners = {
       # City of Boston
       'CITY OF BOSTON BY FCL', 'CITY OF BOSTON',
       'CITY OF BOSTON PARKS AND', 'CITY OF BOSTON PARKS COMM',
       'CITY OF BOSTON TRST', 'CITY OF BOSTON DPW', 'CITY OF BOSTON PWD',
       'CITY OF BOSTON - DND', 'CITY OF BOSTON FCL',
       'CITY OF BOSTON FCL.', 'CITY OF BOSTON (POLICE)',
       'CITY OF BOSTON (POLICE DEPT)', 'CITY OF BOSTON SCHOOL DEPT',
       'CITY OF BOSTON MUNICIPAL CP', 'CITY OF BOSTON PUBLIC FACLTS',
       'CITY OF BOSTON PARKS', 'CITY OF BOSTON MUNICIPAL',
       'CITY OF BOSTON (FCL)', 'CITY OF BOSTON PARKS/REC',
       'CITY OF BOSTON DND', 'CITY OF BOSTON PUB FACIL',
       'CITY OF BOSTON CREDIT UNION', 'CITY OF BOSTON PUBLIC HEALTH',
       'CITY OF BOSTON PARKS & REC', 'CITY OF BOSTON PFD',
       'CITY OF BOSTON CONSERVATION', 'CITY OF BOSTON PUBLIC WORKS',
       'CITY OF BOSTON (PARK)', 'CITY OF BOSTON LIBRARY DEPT', 'CITY OB BOSTON PUBLIC HEALTH','CITY OF BOSTON FIRE DEPARTME', 
       # Boston Water and Sewer Commission
      'BOSTON WATER AND SEWER COMM', 'BOSTON WATER AND SEWER', 'BOSTON WATER & SEWER','BOSTON WATER & SEWER COMM',
       # Boston Housing Authority
      'BOSTON HOUSING AUTHORITY', 'BOSTON HOUSING AUTH', 'BOSTON HOUSING AUTHOURTY',
       # Boston Redevelopment Authority
       'BOSTON REDEVELOPMENT LLC', 'BOSTON REDEVELOPMENT AUTHORI','BOSTON REDEVELOPMENT', 'BOSTON REDEVELOPMENT AUTHOR', 'BOSTON REDEVELOPMNT AUTH',
       'BOSTON REDEVELOPMENT AUTH', 'BOSTON REDEVLPMNT AUTH','BOSTON REDEVLPMNT AUTHOR','BOSTON  REDEVELOPMENT',
       'BOSTON REDEVLOPMENT AUTHORIT', 'BOSTON REDEVOPMENT AUTH', 'BOSTON REDEVLOPMENT AUTH',
       'BOSTON REDEVELOPEMENT AUTH', 'BOSTON REDEVELOP AUTHORITY', 'BOSTON REDEVELPOMENT AUTH',
       # Boston Planning and Development Agency
       'BOSTON DEVELOPMENT'
                 }



# Calculate the total TC_E_A(Tree canopy existing area) i.e. total tree canopy that exists in Boston
total_tree_canopy = parcels_tree_canopy_metrics["TC_E_A"].sum()

# Create a new column called 'baseline_public_or_private' that classifies parcels as public,private, or ROW
parcels_tree_canopy_metrics["baseline_public_or_private"] = parcels_tree_canopy_metrics["OWNER"].apply(lambda x: "Public" if x in public_owners 
                                 else "ROW" if x == "ROW" 
                                 else "Private")

# Group TC_E_A(Tree canopy existing area) by public or private or ROW
baseline_agg_tree_canopy = parcels_tree_canopy_metrics.groupby("baseline_public_or_private")["TC_E_A"].sum().reset_index()

# Add total tree canopy to the dataframe
baseline_agg_tree_canopy["Total Tree Canopy Area"] = total_tree_canopy

# Calculate Percentage of Tree Canopy
baseline_agg_tree_canopy["Percentage of Total Tree Canopy"] = ((baseline_agg_tree_canopy["TC_E_A"] / baseline_agg_tree_canopy["Total Tree Canopy Area"])*100).round(2)

baseline_agg_tree_canopy

,baseline_public_or_private,TC_E_A,Total Tree Canopy Area,Percentage of Total Tree Canopy
0,Private,251104099,357959872,70.15
1,Public,66016522,357959872,18.44
2,ROW,40839251,357959872,11.41


## 2. Using Property Occupancy Codes

This set of steps attempts to use property occupancy codes from [Analyze Boston Property Assessment Data FY2019](https://data.boston.gov/dataset/property-assessment) to determine owners that are tax-exempt by looking at codes that had City of Boston as the owner as a starting point. From there we cross referenced the [Property Occupancy Codes](https://data.boston.gov/dataset/property-assessment/resource/d6c1268c-cd83-4dc3-a914-bba1ed59da6d) to determine any additional codes that seemed to match the entities we determined to be city owned. We left the codes as is, but more codes could be manually included in the future. 

In [ ]:
# identify use codes that imply public 
# start with those that have 'CITY OF BOSTON' as owner 
# list can be appended to for more considerations
cob_use_codes = parcels_tree_canopy_metrics[parcels_tree_canopy_metrics["OWNER"] == "CITY OF BOSTON"]["PTYPE"].unique()
cob_use_codes = np.sort(cob_use_codes.tolist())

# filter the dataframe to parcels with these property codes identified above
filtered_df = parcels_tree_canopy_metrics[parcels_tree_canopy_metrics['PTYPE'].isin(cob_use_codes)]

# Now, we must filter the owners generated by property codes to ensure removal of any federal or state owned properties 
# strings_to_keep is a list of key terms/abbrevations to search for in the OWNERS column
strings_to_keep = ["CITY OF BOSTON", 'BOSTON HOUSING AUTHORITY', "AUTH", "REDEV","DEVELOPMENT","REDEVELOPMENT", "WATER", 
                   "SEWER", "AUTHORITY","COB", "BPRD", "BPDA", "PLANNING", "AGENCY",
                   "HOUSING"]

# function to search OWNERS with partial string matching using str.contains and regex.
def filter_dataframe_by_partial_string(df, column_name, string_list):
  """Filters a dataframe by partial string match"""
  filtered_df = df[df[column_name].str.contains('|'.join(string_list), regex=True)]
  return filtered_df

#returns a dataset with the OWNERS filtered to those that match the keywords in strings_to_keep
filtered_result_partial = filter_dataframe_by_partial_string(filtered_df, 'OWNER', strings_to_keep)

# check the length of unique owners after the filter 
len(filtered_result_partial["OWNER"].unique())

# export the list of unique owners in the filtered dataset to csv for manual checking
# uncomment the below if you'd like to see the original output
# however, we manually edited the csv named 'ptype_classified_owners.csv' to flag any owners that were not necessarily public but contained key words. 
# nulls were considered 'Public' and flagged owners had a '1' in the column 'Flagged'

#pd.DataFrame(filtered_result_partial["OWNER"].unique()).to_csv('classifications/ptype_unfiltered_owner_output.csv', index=False)

37

In [ ]:
# read in the data that has any misclassifications flagged
classified_ptype_owners = pd.read_csv('classifications/ptype_classified_owners.csv')

#fill in na's with 0's
classified_ptype_owners = classified_ptype_owners.fillna({'Flagged': 0})

# convert public owners to a list 
ptype_public_owners = classified_ptype_owners[classified_ptype_owners["Flagged"] == 0]["OWNER"].tolist()
len(ptype_public_owners), ptype_public_owners

(12,
 ['CITY OF BOSTON BY FCL',
  'CITY OF BOSTON',
  'BOSTON REDEVELOPMENT',
  'BOSTON REDEVELOPMENT AUTHOR',
  'CITY OF BOSTON PARKS AND',
  'CITY OF BOSTON PARKS COMM',
  'BOSTON HOUSING AUTHORITY',
  'BOSTON HOUSING AUTH',
  'BOSTON REDEVELPMENT AUTH',
  'BOSTON REDEVELOPMNT AUTH',
  'BOSTON REDEVELOPMENT AUTH',
  'BOSTON WATER & SEWER'])

## 3. Using TheFuzz Package

This method attempts to use the package TheFuzz to determine close matches to our key search terms i.e. determining if any of the owners in the tree canopy dataset are subset strings of the key terms we're looking for. 


In [ ]:
pip install thefuzz

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from thefuzz import fuzz

# List of known owners of public parcels of land
public_entities = ["CITY OF BOSTON",
                   "BOSTON WATER AND SEWER COMMISSION",
                   "BOSTON HOUSING AUTHORITY",
                   "BOSTON PUBLIC SCHOOLS",
                   "BOSTON REDEVELOPMENT AUTHORITY",
                   "BOSTON PLANNING AND DEVELOPMENT AGENCY"
]

# Stores Fuzz matching results
scores = []

# Loop through each owner and calculate best token_set_ratio
for owner in owner_count['OWNER']:
    best_score = max([fuzz.token_set_ratio(owner, term) for term in public_entities])
    scores.append({
        "OWNER": owner,
        "SCORE": best_score
    })

fuzz_unfiltered_owner_output = pd.DataFrame(scores)


# The original output is located in 'classifications/fuzz_unfiltered_owner_output.csv'
# The output was manually edited to flag any non-public owners that were included. 
# This manually edited file is saved to 'classifications/fuzz_classified_owners.csv'
# Column called 'Flagged' was added: Null's represent public ownership, while those that were flagged as private had 1's 

# Export to a csv to check for any misclassifications
fuzz_unfiltered_owner_output[fuzz_unfiltered_owner_output["SCORE"] > 78].to_csv('classifications/fuzz_unfiltered_owner_output.csv')


In [45]:
#after manually flagging errors reload the data
fuzz_classified_owners = pd.read_csv("classifications/fuzz_classified_owners.csv")

#fill in na's with 0's
fuzz_classified_owners = fuzz_classified_owners.fillna({'Flagged': 0})

#drop any non-public owners that were flagged by a 1 in the error column 
fuzz_public_owners = fuzz_classified_owners[fuzz_classified_owners["Flagged"] == 0]["OWNER"].tolist()

#returns list of public owners 
fuzz_public_owners, len(fuzz_public_owners)

(['CITY OF BOSTON',
  'CITY OF BOSTON BY FCL',
  'BOSTON HOUSING AUTHORITY',
  'BOSTON REDEVELOPMENT AUTH',
  'BOSTON HOUSING AUTH',
  'CITY OF BOSTON FCL',
  'BOSTON REDEVELOPMNT AUTH',
  'BOSTON REDVLPMNT AUTH',
  'BOSTON REDEVELOPMENT',
  'BOSTON REDEVLPMNT AUTH',
  'CITY OF BOSTON MUNICIPAL CP',
  'BOSTON REDEVLPMNT AUTHOR',
  'BOSTON WATER & SEWER',
  'BOSTON REDVLPMNT AUTHOR',
  'BOSTON REDEVELPMENT AUTH',
  'BOSTON REDEVELPMNT AUTH',
  'BOSTON WATER & SEWER COMM',
  'BOSTON REDEVELOPMENT LLC',
  'BOSTON DEVELOPMENT',
  'CITY OF BOSTON (POLICE)',
  'CITY OF BOSTON PARKS AND',
  'CITY OF BOSTON PWD',
  'CITY OF BOSTON SCHOOL DEPT',
  'CITY  OF  BOSTON',
  'CITY OF BOSTON PUBLIC HEALTH',
  'BOSTON REDEVELOPMENT AUTHOR',
  'CITY OF BOSTON DND',
  'BOSTON REDEVELOP AUTHORITY',
  'BOSTON REDEVLOPMENT AUTH',
  'CITY OF BOSTON (POLICE DEPT)',
  'CITY OF BOSTON FCL.',
  'CITY OF BOSTON - DND',
  'CITY OF BOSTON CREDIT UNION',
  'CITY OF BOSTON PARKS',
  'CITY OF BOSTON TRST',
  'CITY OF 

In [46]:
# Combine lists of owners from 3 methods and remove duplicates
combined_owner_names = list(set(public_owners) | set(ptype_public_owners) | set(fuzz_public_owners))

combined_owner_names, len(combined_owner_names)

(['CITY OF BOSTON PARKS/REC',
  'BOSTON REDEVELOPMENT AUTHORI',
  'CITY OB BOSTON PUBLIC HEALTH',
  'BOSTON REDEVELOPEMENT AUTH',
  'BOSTON REDEVOPMENT AUTH',
  'CITY OF BOSTON PARKS COMM',
  'CITY OF BOSTON TRST',
  'CITY OF BOSTON PUB FACIL',
  'CITY OF BOSTON PWD',
  'BOSTON REDEVLOPMENT AUTH',
  'CITY OF BOSTON PFD',
  'BOSTON REDEVLOPMENT AUTHORIT',
  'CITY OF BOSTON DND',
  'BOSTON WATER AND SEWER COMM',
  'BOSTON  REDEVELOPMENT',
  'BOSTON REDEVELOP AUTHORITY',
  'BOSTON WATER & SEWER COMM',
  'BOSTON REDEVELPMNT AUTH',
  'CITY OF BOSTON PARKS AND',
  'CITY OF BOSTON SCHOOL DEPT',
  'CITY  OF  BOSTON',
  'CITY OF BOSTON CREDIT UNION',
  'CITY OF BOSTON DPW',
  'BOSTON DEVELOPMENT',
  'CITY OF BOSTON',
  'CITY OF BOSTON BY FCL',
  'CITY OF BOSTON PUBLIC FACLTS',
  'BOSTON REDEVELOPMENT AUTHOR',
  'BOSTON REDEVELOPMENT',
  'CITY OF BOSTON CONSERVATION',
  'CITY OF BOSTON (PARK)',
  'BOSTON HOUSING AUTH',
  'BOSTON REDEVELOPMNT AUTH',
  'BOSTON REDEVLPMNT AUTHOR',
  'CITY OF BOSTON

In [47]:
# convert list of combined owners to a csv file 
pd.DataFrame(combined_owner_names, columns=["OWNER"]).to_csv("classifications/combined_list_of_public_owners.csv")

In [48]:
# relabel all parcels in the dataset to be either public or private based on ownership
# any parcels with owners included in the combined_owner_names list were considered public, all else was considered Private
# ROW data was excluded for consistency since there are public and private roads/sidewalks/streets
parcels_tree_canopy_metrics["public_or_private"] = parcels_tree_canopy_metrics["OWNER"].apply(lambda x: 
                                "Public" if x in combined_owner_names
                                 else "ROW" if x == "ROW" 
                                 else "Private")


# Group TC_E_A(Tree canopy existing area) by public/private/row ownership
agg_tree_canopy = parcels_tree_canopy_metrics.groupby("public_or_private")["TC_E_A"].sum().reset_index()

# Add total tree canopy to the dataframe
agg_tree_canopy["Total Tree Canopy Area"] = parcels_tree_canopy_metrics["TC_E_A"].sum()

# Calculate Percentage of Tree Canopy
agg_tree_canopy["Percentage of Total Tree Canopy"] = ((agg_tree_canopy["TC_E_A"] / agg_tree_canopy["Total Tree Canopy Area"])*100).round(2)

# export this table to be a csv for visualizing 
agg_tree_canopy.to_csv("visualization_datasets/pub_priv_row.csv")

agg_tree_canopy


,public_or_private,TC_E_A,Total Tree Canopy Area,Percentage of Total Tree Canopy
0,Private,251006691,357959872,70.12
1,Public,66113930,357959872,18.47
2,ROW,40839251,357959872,11.41


# What are the trends in tree canopy loss or gain (2014-2019) for public and private areas?

### Dataset: [Parcel Tree Canopy Change Metrics](https://data.boston.gov/dataset/canopy-change-assessment-tree-canopy-change-metrics)

In [55]:
# load in the dataset
parcel_canopy_change_metrics = pd.read_csv("datasets/PARCELS_Tree_Canopy_Change_Metrics.csv")

# Replace any rows with a ' '  owner with a null
parcel_canopy_change_metrics["OWNER"] = parcel_canopy_change_metrics["OWNER"].replace(" ", pd.NA)

# find all null rows
parcel_canopy_change_metrics[parcel_canopy_change_metrics["OWNER"].isna()]

# Classify the null row owner as 'ROW', since it represents all roads and sidewalks in Boston
parcel_canopy_change_metrics.loc[parcel_canopy_change_metrics["OWNER"].isna(), "OWNER"] = "ROW"

# apply the classification method to the owners column
parcel_canopy_change_metrics["public_or_private"] = parcel_canopy_change_metrics["OWNER"].apply(lambda x: "Public" if x in combined_owner_names
                                 else "ROW" if x == "ROW" 
                                 else "Private")


/var/folders/3k/5zd1hcfd4jv0bjvll1h2r7kr0000gn/T/ipykernel_52958/1254337679.py:2: DtypeWarning: Columns (4,6,12,14,18,27) have mixed types. Specify dtype option on import or set low_memory=False.
  parcel_canopy_change_metrics = pd.read_csv("datasets/PARCELS_Tree_Canopy_Change_Metrics.csv")


In [56]:
parcel_canopy_change_metrics.columns

Index(['FID', 'FID_Boston', 'BOSTON_LAN', 'FID_COB_FY', 'PID', 'CM_ID',
       'GIS_ID', 'OWNER', 'ST_NUM', 'ST_NAME', 'ST_NAME_SU', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'PTYPE', 'LOTSIZE', 'MAIL_ADDRE', 'MAIL_CS',
       'MAIL_ZIP', 'OWN_OCC', 'GROSS_AREA', 'BLDG_AREA', 'AV_TOTAL', 'AV_LAND',
       'AV_BLDG', 'GROSS_TAX', 'NUM_FLOORS', 'PID_LONG', 'FULL_ADDRE', 'CITY',
       'STArea__', 'STLength__', 'Shape_STAr', 'Shape_STLe', 'LU_GENERAL',
       'TC_ID', 'Shape_Leng', 'OBJECTID', 'TC_ID_1', 'LandArea', 'Gain',
       'Loss', 'No_Change', 'TreeCanopy', 'TreeCano_1', 'Change_Are',
       'Change_Per', 'TreeCano_2', 'TreeCano_3', 'Change_P_1', 'Shape__Area',
       'Shape__Length', 'public_or_private'],
      dtype='object')

### Column Definitions:
* TreeCanopy= 2014 total canopy area (baseline)
* TreeCano_1 = 2019 total canopy area

In [57]:
#EDA
parcel_canopy_change_metrics["diff"] = parcel_canopy_change_metrics['TreeCano_1'] - parcel_canopy_change_metrics['TreeCanopy'] 

parcel_canopy_change_metrics["diff_g_l"] = parcel_canopy_change_metrics['Gain'] - parcel_canopy_change_metrics['Loss'] 

parcel_canopy_change_metrics[["Gain", "Loss","No_Change", 'TreeCanopy', 'TreeCano_1', 'diff', "diff_g_l"]]

,Gain,Loss,No_Change,TreeCanopy,TreeCano_1,diff,diff_g_l
0,7.443209e+06,6.432831e+06,3.343496e+07,3.986780e+07,4.087817e+07,1.010378e+06,1.010378e+06
1,4.220522e+00,0.000000e+00,7.893076e-01,7.893076e-01,5.009830e+00,4.220522e+00,4.220522e+00
2,0.000000e+00,2.350375e+00,0.000000e+00,2.350375e+00,0.000000e+00,-2.350375e+00,-2.350375e+00
3,3.760469e+00,2.414919e+01,3.796573e+00,2.794576e+01,7.557041e+00,-2.038872e+01,-2.038872e+01
4,4.944661e+00,0.000000e+00,9.597758e-01,9.597758e-01,5.904437e+00,4.944661e+00,4.944661e+00
...,...,...,...,...,...,...,...
173870,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
173871,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
173872,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
173873,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [59]:
#aggregate tree canopy gain and loss by ownership
gain_loss_agg = parcel_canopy_change_metrics.groupby("public_or_private")[["Gain", "Loss",'TreeCanopy', 'TreeCano_1', "diff_g_l"]].sum().reset_index()

#rename the columns 
gain_loss_agg = gain_loss_agg.rename(columns={"TreeCanopy": "2014 Tree Canopy", "TreeCano_1": "2019 Tree Canopy"})

#calculate the percentage difference in tree canopy from 2014 to 2019
gain_loss_agg["Percent_Diff"] = ((gain_loss_agg["2019 Tree Canopy"] -  gain_loss_agg["2014 Tree Canopy"])/gain_loss_agg["2014 Tree Canopy"]) *100

#export dataset for visualization
gain_loss_agg.to_csv("visualization_datasets/gain_loss_agg.csv")

gain_loss_agg

,public_or_private,Gain,Loss,2014 Tree Canopy,2019 Tree Canopy,diff_g_l,Percent_Diff
0,Private,2.716149e+07,2.920234e+07,2.531463e+08,2.511055e+08,-2.040853e+06,-0.806195
1,Public,5.468686e+06,4.190453e+06,6.486015e+07,6.613838e+07,1.278234e+06,1.970753
2,ROW,7.443209e+06,6.432831e+06,3.986780e+07,4.087817e+07,1.010378e+06,2.534321


# Which areas have the highest potential for expanding tree canopy, and are they public or private spaces?

* TC_PV_A= Possible vegetation area. Grass or shrub area that is theoretically available for
the establishment of tree canopy.
* TC_Pi_A= Possible impervious area. Asphalt or concrete surfaces or bare soil, excluding
roads and buildings, that are theoretically available for the establishment of tree canopy.
* TC_P_A= Possible area. Area theoretically available for establishment of tree canopy.
* TC_Land_A = Land area. Land area excluding water bodies.

In [60]:
parcels_tree_canopy_metrics.columns

Index(['FID', 'FID_Boston', 'BOSTON_LAN', 'FID_COB_FY', 'PID', 'CM_ID',
       'GIS_ID', 'OWNER', 'ST_NUM', 'ST_NAME', 'ST_NAME_SU', 'UNIT_NUM',
       'ZIPCODE', 'LU', 'PTYPE', 'LOTSIZE', 'MAIL_ADDRE', 'MAIL_CS',
       'MAIL_ZIP', 'OWN_OCC', 'GROSS_AREA', 'BLDG_AREA', 'AV_TOTAL', 'AV_LAND',
       'AV_BLDG', 'GROSS_TAX', 'NUM_FLOORS', 'PID_LONG', 'FULL_ADDRE', 'CITY',
       'STArea__', 'STLength__', 'Shape_STAr', 'Shape_STLe', 'LU_GENERAL',
       'TC_ID', 'Shape_Leng', 'OBJECTID', 'TC_ID_1', 'Total_A', 'Can_A',
       'Grass_A', 'Soil_A', 'Water_A', 'Build_A', 'Road_A', 'Paved_A',
       'Perv_A', 'Imperv_A', 'Can_P', 'Grass_P', 'Soil_P', 'Water_P',
       'Build_P', 'Road_P', 'Paved_P', 'Perv_P', 'Imperv_P', 'OBJECTID_1',
       'TC_ID_12', 'VALUE_0', 'TC_E_A', 'TC_Pv_A', 'TC_Land_A', 'TC_Pi_A',
       'TC_P_A', 'TC_E_P', 'TC_Pv_P', 'TC_P_P', 'TC_Pi_P', 'Shape__Area',
       'Shape__Length', 'Parcel', 'baseline_public_or_private',
       'public_or_private'],
      dtype='object')

In [61]:
potential_expansion = parcels_tree_canopy_metrics.groupby("public_or_private")[["TC_Pv_A", "TC_Pi_A",'TC_P_A', "TC_Land_A"]].sum().reset_index()

potential_expansion["Percentage Possible Vegetation Area"] = potential_expansion["TC_Pv_A"]/potential_expansion["TC_Land_A"]

potential_expansion["Percentage Possible Impervious Area"] = potential_expansion["TC_Pi_A"]/potential_expansion["TC_Land_A"]

potential_expansion["Percentage Possible Tree Canopy Area"] = round((potential_expansion["TC_P_A"]/potential_expansion["TC_Land_A"])*100)

potential_expansion[["public_or_private", "Percentage Possible Tree Canopy Area"]]

,public_or_private,Percentage Possible Tree Canopy Area
0,Private,41.0
1,Public,47.0
2,ROW,20.0


# How does tree canopy coverage vary across neighborhoods and land use types?

In [62]:
# Replace any rows with a ' '  owner with a null
parcels_tree_canopy_metrics["LU_GENERAL"] = parcels_tree_canopy_metrics["LU_GENERAL"].replace(" ", pd.NA)
parcels_tree_canopy_metrics[parcels_tree_canopy_metrics["LU_GENERAL"].isna()]
# Classify the null row owner as 'ROW', since it represents all roads and sidewalks in Boston
parcels_tree_canopy_metrics.loc[parcels_tree_canopy_metrics["LU_GENERAL"].isna(), "LU_GENERAL"] = "ROW"

In [ ]:
# aggregate tree canopy area by land use
lu_agg = parcels_tree_canopy_metrics.groupby('LU_GENERAL')['TC_E_A'].sum().reset_index()

#calculate the total amount of tree canopy 
lu_agg["total"] = parcels_tree_canopy_metrics["TC_E_A"].sum()

#calcualte the percentage of total tree canopy that exists in each land use area
lu_agg["Percentage of Total Tree Canopy"] = round((lu_agg["TC_E_A"]/lu_agg["total"])*100, 2)

lu_agg

,LU_GENERAL,TC_E_A,total,Percentage of Total Tree Canopy
0,Ag,987982,357959872,0.28
1,Commercial,10383144,357959872,2.90
2,Industrial,1840747,357959872,0.51
3,Institutional,160606226,357959872,44.87
4,Mixed Use,5228428,357959872,1.46
5,ROW,40839251,357959872,11.41
6,Residential,138074094,357959872,38.57


In [66]:
#read in the tree canopy metrics data aggregated by neighborhood
neighborhoods_tree_canopy = pd.read_csv('datasets/Neighborhood_Tree_Canopy_Metrics.csv')

In [67]:
#aggregate the total tree canopy area by neighborhood 'Name'
neigh_agg = neighborhoods_tree_canopy.groupby('Name')['TC_E_A'].sum().reset_index()

#Find the total tree canopy in the dataset
neigh_agg["total"] = neigh_agg["TC_E_A"].sum()

#calculate the percentage of total tree canopy in each neighborhood
neigh_agg["Percentage of Total Tree Canopy"] = (neigh_agg["TC_E_A"]/neigh_agg["total"])*100

#sort the neighborhoods by the 'Percentage of Total Tree Canopy
neigh_agg_sorted = neigh_agg.sort_values(by='Percentage of Total Tree Canopy', ascending=False)

neigh_agg_sorted

,Name,TC_E_A,total,Percentage of Total Tree Canopy
16,West Roxbury,64295095,357771079,17.971015
8,Hyde Park,51423243,357771079,14.373225
9,Jamaica Plain,49183173,357771079,13.747107
4,Dorchester,48018572,357771079,13.421591
0,Allston-Brighton,27310199,357771079,7.633428
12,Roslindale,24844503,357771079,6.944246
10,Mattapan,20799757,357771079,5.813706
13,Roxbury,19118383,357771079,5.343747
5,East Boston,9609091,357771079,2.685821
7,Harbor Islands,8968553,357771079,2.506785
